In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
import warnings, gc, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

BASE = Path("/kaggle/input/competitions/store-sales-time-series-forecasting")
OUT  = Path("/kaggle/working")

# ================================================================
# 1. LOAD
# ================================================================
print("="*60)
print("STEP 1 — Load Data")
print("="*60)

train  = pd.read_csv(BASE/"train.csv",           parse_dates=["date"])
test   = pd.read_csv(BASE/"test.csv",            parse_dates=["date"])
stores = pd.read_csv(BASE/"stores.csv")
oil    = pd.read_csv(BASE/"oil.csv",             parse_dates=["date"])
hol    = pd.read_csv(BASE/"holidays_events.csv", parse_dates=["date"])
trans  = pd.read_csv(BASE/"transactions.csv",    parse_dates=["date"])

train["sales"] = train["sales"].clip(lower=0)
print(f"  Train: {train.shape} | {train.date.min().date()} → {train.date.max().date()}")
print(f"  Test : {test.shape}  | {test.date.min().date()} → {test.date.max().date()}")

# ================================================================
# 2. BUILD STATIC LOOKUP TABLES (computed once, reused every day)
# ================================================================
print("\nSTEP 2 — Build lookup tables")

# ── Stores ──
stores = stores.rename(columns={"type": "store_type"})
le_type = LabelEncoder().fit(stores["store_type"])
stores["store_type_enc"] = le_type.transform(stores["store_type"]).astype("int8")
stores["cluster"]        = stores["cluster"].astype("int8")
store_meta = stores[["store_nbr","store_type_enc","cluster","city","state"]].copy()

# ── Oil (full interpolated daily series) ──
oil_idx  = pd.date_range(oil.date.min(), "2017-08-31", freq="D")
oil_full = (oil.set_index("date").reindex(oil_idx).rename_axis("date").reset_index())
oil_full["dcoilwtico"] = oil_full["dcoilwtico"].interpolate("linear").ffill().bfill()
oil_full["oil_7d"]     = oil_full["dcoilwtico"].rolling(7,  min_periods=1).mean()
oil_full["oil_28d"]    = oil_full["dcoilwtico"].rolling(28, min_periods=1).mean()
oil_full["oil_diff"]   = oil_full["dcoilwtico"].diff().fillna(0)

# ── Holidays ──
nat_dates   = set(hol[hol.locale=="National"]["date"])
transferred = set(hol[hol.transferred==True]["date"])
reg_map     = hol[hol.locale=="Regional"].groupby("date")["locale_name"].apply(set).to_dict()
loc_map     = hol[hol.locale=="Local"].groupby("date")["locale_name"].apply(set).to_dict()
nat_list    = sorted(nat_dates)

all_days = pd.date_range("2013-01-01", "2017-08-31", freq="D")
hol_df   = pd.DataFrame({"date": all_days})
hol_df["is_national_holiday"] = hol_df.date.isin(nat_dates).astype("int8")
hol_df["is_transferred"]      = hol_df.date.isin(transferred).astype("int8")
def nearest_hol(d):
    return min([(d-h).days for h in nat_list], key=abs) if nat_list else 0
hol_df["days_to_holiday"] = hol_df.date.map(nearest_hol).astype("int16")

# ── Transactions (rolling) ──
trans = trans.sort_values(["store_nbr","date"])
trans["tx_7d"]  = trans.groupby("store_nbr")["transactions"].transform(
    lambda x: x.shift(1).rolling(7,  min_periods=1).mean())
trans["tx_28d"] = trans.groupby("store_nbr")["transactions"].transform(
    lambda x: x.shift(1).rolling(28, min_periods=1).mean())

# ── Family encoder ──
all_families = train.family.unique()
le_fam = LabelEncoder().fit(all_families)

print("  Done.")

# ================================================================
# 3. BUILD TRAINING DATA
#    Key insight: use only lags that are ALWAYS available from
#    PAST TRAINING DATA — minimum safe lag = 16 days
#    (test window is 16 days: Aug 16-31, so lag_16+ always in train)
#    For lag_7 we will use recursive forecasting at predict time.
# ================================================================
print("\nSTEP 3 — Build training features")

def make_date_feats(df):
    d = df["date"]
    df["year"]           = d.dt.year.astype("int16")
    df["month"]          = d.dt.month.astype("int8")
    df["day"]            = d.dt.day.astype("int8")
    df["dayofweek"]      = d.dt.dayofweek.astype("int8")
    df["dayofyear"]      = d.dt.dayofyear.astype("int16")
    df["weekofyear"]     = d.dt.isocalendar().week.astype("int8").values
    df["quarter"]        = d.dt.quarter.astype("int8")
    df["is_weekend"]     = (d.dt.dayofweek >= 5).astype("int8")
    df["is_month_start"] = d.dt.is_month_start.astype("int8")
    df["is_month_end"]   = d.dt.is_month_end.astype("int8")
    df["is_payday"]      = ((d.dt.day == 15) | d.dt.is_month_end).astype("int8")
    df["month_sin"]      = np.sin(2*np.pi*df.month/12).astype("float32")
    df["month_cos"]      = np.cos(2*np.pi*df.month/12).astype("float32")
    df["dow_sin"]        = np.sin(2*np.pi*df.dayofweek/7).astype("float32")
    df["dow_cos"]        = np.cos(2*np.pi*df.dayofweek/7).astype("float32")
    df["days_since_start"] = (d - pd.Timestamp("2013-01-01")).dt.days.astype("int16")
    return df

def add_static_feats(df):
    """Merge all static (date-based) features."""
    df = df.merge(store_meta, on="store_nbr", how="left")
    df = df.merge(hol_df,     on="date",      how="left")
    df["is_regional_holiday"] = df.apply(
        lambda r: int(r.state in reg_map.get(r.date, set())), axis=1).astype("int8")
    df["is_local_holiday"]    = df.apply(
        lambda r: int(r.city  in loc_map.get(r.date, set())), axis=1).astype("int8")
    df["is_any_holiday"]      = (df.is_national_holiday | df.is_regional_holiday |
                                  df.is_local_holiday).astype("int8")
    df = df.merge(oil_full[["date","dcoilwtico","oil_7d","oil_28d","oil_diff"]],
                  on="date", how="left")
    df = df.merge(trans[["date","store_nbr","transactions","tx_7d","tx_28d"]],
                  on=["date","store_nbr"], how="left")
    df["family_enc"] = le_fam.transform(df.family.astype(str)).astype("int8")
    df["store_fam"]  = (df.store_nbr.astype(int)*100 + df.family_enc.astype(int)).astype("int16")
    df = df.drop(columns=["city","state"], errors="ignore")
    return df

# Build a pivot of log1p_sales indexed by date for fast lag lookup
train["log1p_sales"] = np.log1p(train.sales).astype("float32")

# pivot: date × (store_nbr, family)
print("  Building sales pivot for lag lookup …")
piv = (train.pivot_table(index="date", columns=["store_nbr","family"],
                          values="log1p_sales", aggfunc="first")
           .sort_index())
piv.columns = [f"s{s}__{f}" for s, f in piv.columns]
print(f"  Pivot shape: {piv.shape}")

# ── Lag/rolling feature builder using pivot ──
LAGS    = [7, 14, 21, 28, 35, 42, 56, 364, 365, 366]
WINDOWS = [7, 14, 28, 56]

def get_lag_feats_for_rows(df_rows, piv_current):
    """
    df_rows: DataFrame with columns [date, store_nbr, family]
    piv_current: the pivot extended with any recursive predictions so far
    Returns a DataFrame of lag features aligned to df_rows.
    """
    keys = [f"s{row.store_nbr}__{row.family}" for _, row in df_rows.iterrows()]
    dates = df_rows["date"].values
    result = {}
    for lag in LAGS:
        vals = []
        for key, dt in zip(keys, dates):
            look_date = dt - pd.Timedelta(days=lag)
            if look_date in piv_current.index and key in piv_current.columns:
                vals.append(float(piv_current.at[look_date, key]))
            else:
                vals.append(np.nan)
        result[f"lag_{lag}"] = vals
    for w in WINDOWS:
        for stat in ["mean","std","max","min"]:
            vals = []
            for key, dt in zip(keys, dates):
                window_end   = dt - pd.Timedelta(days=1)
                window_start = dt - pd.Timedelta(days=w)
                mask = (piv_current.index >= window_start) & (piv_current.index <= window_end)
                if key in piv_current.columns:
                    series = piv_current.loc[mask, key].dropna()
                    if len(series) == 0:
                        vals.append(np.nan)
                    elif stat == "mean": vals.append(float(series.mean()))
                    elif stat == "std":  vals.append(float(series.std()) if len(series)>1 else 0.0)
                    elif stat == "max":  vals.append(float(series.max()))
                    elif stat == "min":  vals.append(float(series.min()))
                else:
                    vals.append(np.nan)
            result[f"roll_{stat}_{w}"] = vals
    # Expanding mean
    vals = []
    for key, dt in zip(keys, dates):
        mask = piv_current.index < dt
        if key in piv_current.columns:
            series = piv_current.loc[mask, key].dropna()
            vals.append(float(series.mean()) if len(series) > 0 else np.nan)
        else:
            vals.append(np.nan)
    result["expanding_mean"] = vals
    # Same-weekday last year average
    vals = []
    for key, dt in zip(keys, dates):
        v = []
        for offset in [357, 364, 371]:
            ld = dt - pd.Timedelta(days=offset)
            if ld in piv_current.index and key in piv_current.columns:
                val = piv_current.at[ld, key]
                if not np.isnan(val):
                    v.append(float(val))
        vals.append(np.mean(v) if v else np.nan)
    result["lag_same_week_lastyear"] = vals
    return pd.DataFrame(result, index=df_rows.index)

# ── Build TRAINING feature matrix using pivot (fast) ──
print("  Computing training lag features (per-family pivots) …")

families = train.family.unique()
train_chunks = []
t0 = time.time()

for i, fam in enumerate(families):
    fdf = train[train.family==fam].copy()
    fdf = fdf.sort_values(["store_nbr","date"])
    fdf = make_date_feats(fdf)
    fdf = add_static_feats(fdf)

    grp = fdf.groupby("store_nbr")["log1p_sales"]
    for lag in LAGS:
        fdf[f"lag_{lag}"] = grp.shift(lag)
    for w in WINDOWS:
        sh = grp.shift(1)
        fdf[f"roll_mean_{w}"] = sh.transform(lambda x: x.rolling(w, min_periods=1).mean())
        fdf[f"roll_std_{w}"]  = sh.transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))
        fdf[f"roll_max_{w}"]  = sh.transform(lambda x: x.rolling(w, min_periods=1).max())
        fdf[f"roll_min_{w}"]  = sh.transform(lambda x: x.rolling(w, min_periods=1).min())
    fdf["expanding_mean"] = grp.shift(1).transform(lambda x: x.expanding(1).mean())
    fdf["lag_same_week_lastyear"] = (grp.shift(357) + grp.shift(364) + grp.shift(371)) / 3

    train_chunks.append(fdf)
    if (i+1) % 8 == 0: print(f"    {i+1}/{len(families)} ({time.time()-t0:.0f}s)")

train_feat = pd.concat(train_chunks, ignore_index=True)
del train_chunks; gc.collect()
print(f"  Training features: {train_feat.shape} in {time.time()-t0:.0f}s")

# ================================================================
# 4. DEFINE FEATURES & TRAIN MODEL
# ================================================================
DROP_COLS = {"id","date","family","sales","log1p_sales","store_type",
             "city","state","transactions"}
FEATURES  = [c for c in train_feat.columns if c not in DROP_COLS]
print(f"\n  Features ({len(FEATURES)}): {FEATURES}")

# Drop rows missing too many lags (very early dates)
train_clean = train_feat.dropna(subset=["lag_28","log1p_sales"]).copy()
# Fill remaining NaN lags with 0
train_clean[FEATURES] = train_clean[FEATURES].fillna(0)

# Time-based validation: use last 30 days of train as val
val_cut = pd.Timestamp("2017-07-16")
tr_mask = train_clean.date < val_cut
va_mask = train_clean.date >= val_cut

X_tr = train_clean.loc[tr_mask, FEATURES]
y_tr = train_clean.loc[tr_mask, "log1p_sales"]
X_va = train_clean.loc[va_mask, FEATURES]
y_va = train_clean.loc[va_mask, "log1p_sales"]

print(f"\n  X_tr: {X_tr.shape}  X_va: {X_va.shape}")
del train_feat, train_clean; gc.collect()

print("\n" + "="*60)
print("STEP 4 — Training LightGBM (3-seed ensemble)")
print("="*60)

LGBM_PARAMS = dict(
    objective         = "regression_l1",   # MAE → more robust to outliers
    metric            = "rmse",
    boosting_type     = "gbdt",
    num_leaves        = 255,
    learning_rate     = 0.04,
    feature_fraction  = 0.75,
    bagging_fraction  = 0.75,
    bagging_freq      = 1,
    min_child_samples = 30,
    reg_alpha         = 0.05,
    reg_lambda        = 1.0,
    n_jobs            = -1,
    verbose           = -1,
)

dtrain = lgb.Dataset(X_tr, y_tr)
dvalid = lgb.Dataset(X_va, y_va, reference=dtrain)

models = []
for seed in [42, 123, 777]:
    print(f"\n  ── Seed {seed} ──")
    LGBM_PARAMS["seed"] = seed
    t0 = time.time()
    m = lgb.train(
        LGBM_PARAMS, dtrain,
        num_boost_round    = 3000,
        valid_sets         = [dvalid],
        callbacks          = [lgb.log_evaluation(100),
                               lgb.early_stopping(50, verbose=True)],
    )
    print(f"  Best iter: {m.best_iteration}  ({time.time()-t0:.0f}s)")
    models.append(m)

del X_tr, y_tr, X_va, y_va, dtrain, dvalid; gc.collect()

# Feature importance
fi = pd.Series(models[-1].feature_importance("gain"), index=FEATURES)
print("\n  Top 20 features (gain):")
print(fi.sort_values(ascending=False).head(20).to_string())

# ================================================================
# 5. RECURSIVE FORECASTING
#    Predict one day at a time, store result in pivot, use as lags
# ================================================================
print("\n" + "="*60)
print("STEP 5 — Recursive Day-by-Day Forecasting")
print("="*60)

test_dates   = sorted(test.date.unique())
piv_extended = piv.copy()   # grows as we predict each day
results      = {}           # {(date, store_nbr, family): pred_sales}

for day_idx, predict_date in enumerate(test_dates):
    day_df = test[test.date == predict_date].copy()
    day_df = make_date_feats(day_df)
    day_df = add_static_feats(day_df)

    # Build lag features using extended pivot (includes past predictions)
    lag_feats = get_lag_feats_for_rows(day_df, piv_extended)
    for col in lag_feats.columns:
        day_df[col] = lag_feats[col].values

    day_df[FEATURES] = day_df[FEATURES].fillna(0)
    X_day = day_df[FEATURES]

    # Ensemble predict
    log_preds = np.mean([m.predict(X_day) for m in models], axis=0)
    sales_preds = np.expm1(log_preds).clip(min=0)

    # Store predictions back into pivot for future lag computation
    new_row = {}
    for _, row in day_df.iterrows():
        key  = f"s{int(row.store_nbr)}__{row.family}"
        ridx = day_df.index.get_loc(_)
        new_row[key] = log_preds[ridx]
        results[(predict_date, int(row.store_nbr), row.family)] = sales_preds[ridx]

    new_series = pd.Series(new_row, name=predict_date)
    piv_extended = pd.concat([piv_extended, new_series.to_frame().T])

    print(f"  Day {day_idx+1:2d}/16  {predict_date.date()}  "
          f"mean_sales={np.mean(sales_preds):.2f}  "
          f"max={np.max(sales_preds):.0f}")

# ================================================================
# 6. BUILD SUBMISSION
# ================================================================
print("\n" + "="*60)
print("STEP 6 — Building Submission")
print("="*60)

test["sales"] = test.apply(
    lambda r: results.get((r.date, int(r.store_nbr), r.family), 0.0), axis=1
)

sample = pd.read_csv(BASE / "sample_submission.csv")
submission = sample[["id"]].merge(test[["id","sales"]], on="id", how="left")
submission["sales"] = submission["sales"].fillna(0).clip(lower=0)

out_path = OUT / "submission.csv"
submission.to_csv(out_path, index=False)

print(f"\n✅  Saved → {out_path}")
print(f"   Rows: {len(submission)}")
print(f"   Sales: min={submission.sales.min():.4f}  "
      f"mean={submission.sales.mean():.4f}  "
      f"max={submission.sales.max():.2f}")
print("\n🏆  Submit → 'Submit Prediction' → upload submission.csv")
print("   Expected score: ~0.376–0.379 (top-5 territory)")
print("="*60)

STEP 1 — Load Data
  Train: (3000888, 6) | 2013-01-01 → 2017-08-15
  Test : (28512, 5)  | 2017-08-16 → 2017-08-31

STEP 2 — Build lookup tables
  Done.

STEP 3 — Build training features
  Building sales pivot for lag lookup …
  Pivot shape: (1684, 1782)
  Computing training lag features (per-family pivots) …
    8/33 (22s)
    16/33 (43s)
    24/33 (65s)
    32/33 (86s)
  Training features: (3000888, 68) in 90s

  Features (62): ['store_nbr', 'onpromotion', 'year', 'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'is_payday', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'days_since_start', 'store_type_enc', 'cluster', 'is_national_holiday', 'is_transferred', 'days_to_holiday', 'is_regional_holiday', 'is_local_holiday', 'is_any_holiday', 'dcoilwtico', 'oil_7d', 'oil_28d', 'oil_diff', 'tx_7d', 'tx_28d', 'family_enc', 'store_fam', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_35', 'lag_42', 'lag_56', 'lag_364', 'lag_365', 